In [0]:
from pyspark.sql import functions as F

base = "workspace.retail_pipeline"

df_fi = spark.read.table(f"{base}.orders_finland_silver")
df_se = spark.read.table(f"{base}.orders_sweden_silver")
df_de = spark.read.table(f"{base}.orders_germany_silver")

In [0]:
def clean_common(df):
    # Normalize column names
    df = df.toDF(*[
        c.strip().lower() for c in df.columns
    ])

    # Clean customer names
    df = df.withColumn(
        "customer_name",
        F.initcap(F.trim(F.col("customer_name")))
    )

    # Ensure correct data types
    df = (
        df.withColumn("customer_id", F.col("customer_id").cast("long"))
          .withColumn("amount", F.col("amount").cast("double"))
    )

    return df

In [0]:
def prep_finland(df):
    df = clean_common(df)

    return (
        df.withColumn(
            "is_large_order",
            F.col("amount") >= 200
        )
        .withColumn(
            "amount_with_vat",
            F.round(F.col("amount") * 1.255, 2)
        )
    )

def prep_sweden(df):
    df = clean_common(df)

    return (
        df.withColumn(
            "order_category",
            F.when(F.col("amount") >= 100, "High")
             .otherwise("Regular")
        )
    )

def prep_germany(df):
    df = clean_common(df)

    return (
        df.withColumn(
            "is_completed",
            F.col("status") == "completed"
        )
    )

In [0]:
df_fi_gold = prep_finland(df_fi)
df_se_gold = prep_sweden(df_se)
df_de_gold = prep_germany(df_de)

In [0]:
df_fi_gold.write.mode("overwrite").saveAsTable(
    f"{base}.orders_finland_gold"
)

df_se_gold.write.mode("overwrite").saveAsTable(
    f"{base}.orders_sweden_gold"
)

df_de_gold.write.mode("overwrite").saveAsTable(
    f"{base}.orders_germany_gold"
)